# **Day 47: Object Oriented Programming (Part 7) - Polymorphism**

**Polymorphism** means **"many forms"**. The same method name can behave **differently** for different objects.

**Real-life example:** The word "open" means different things for a door, a book, and a bank account, but we use the same word for all of them.

We have been using polymorphism from the start without knowing its name:

In [1]:
print(len("Kolkata"), len([1, 2, 3]), len({"a": 1}))    # Same function, different types
print(3 + 4, "3" + "4", [3] + [4])                       # Same + sign, different results

7 3 1
7 34 [3, 4]


## **Polymorphism with Classes (Method Overriding)**
Different payment methods all have a `pay()` method, but each one works differently.

In [2]:
class Payment:
    def pay(self, amount):
        raise NotImplementedError("Every payment type must write its own pay()")

class UPI(Payment):
    def __init__(self, upi_id):
        self.upi_id = upi_id
    def pay(self, amount):
        return f"Paid Rs {amount} using UPI ({self.upi_id})"

class Cash(Payment):
    def pay(self, amount):
        notes = [500, 200, 100, 50, 20, 10]
        used = []
        for n in notes:
            count, amount = divmod(amount, n)
            if count:
                used.append(f"{count}x{n}")
        return "Paid cash: " + ", ".join(used) + (f" + Rs {amount} in coins" if amount else "")

class Card(Payment):
    def __init__(self, last4):
        self.last4 = last4
    def pay(self, amount):
        return f"Paid Rs {amount} with card ending {self.last4}"

bill = 1375
for method in [UPI("tublu@okbank"), Cash(), Card("4242")]:
    print(method.pay(bill))

Paid Rs 1375 using UPI (tublu@okbank)
Paid cash: 2x500, 1x200, 1x100, 1x50, 1x20 + Rs 5 in coins
Paid Rs 1375 with card ending 4242


The loop does not care **what type** each object is. It just calls `.pay()`, and each object does its own thing. This makes the code very easy to extend: to add a new payment type, we only write a new class.

## **Example from Remote Sensing: Different Satellites, Same Method**
Each satellite names its bands differently:
- In **Landsat 8**, Red is band **4** and NIR is band **5**.
- In **Sentinel-2**, Red is band **B4** and NIR is band **B8**.
- In **MODIS**, Red is band **1** and NIR is band **2**.

With polymorphism, every satellite class has the same method `ndvi_bands()`, so the rest of our program does not need to remember these differences.

In [3]:
class Landsat8:
    name = "Landsat 8"
    def ndvi_bands(self):
        return {"red": "B4", "nir": "B5"}
    def resolution(self):
        return 30

class Sentinel2:
    name = "Sentinel-2"
    def ndvi_bands(self):
        return {"red": "B4", "nir": "B8"}
    def resolution(self):
        return 10

class Modis:
    name = "MODIS"
    def ndvi_bands(self):
        return {"red": "Band 1", "nir": "Band 2"}
    def resolution(self):
        return 250

def plan_ndvi(sensor):
    b = sensor.ndvi_bands()
    return f"{sensor.name:<11}: use {b['red']} (Red) and {b['nir']} (NIR), pixel {sensor.resolution()} m"

for s in [Landsat8(), Sentinel2(), Modis()]:
    print(plan_ndvi(s))

Landsat 8  : use B4 (Red) and B5 (NIR), pixel 30 m
Sentinel-2 : use B4 (Red) and B8 (NIR), pixel 10 m
MODIS      : use Band 1 (Red) and Band 2 (NIR), pixel 250 m


The function `plan_ndvi()` works with **any** satellite object, as long as it has `ndvi_bands()` and `resolution()` methods.

## **Duck Typing**
"**If it walks like a duck and quacks like a duck, then it is a duck.**"

In Python, the objects **do not even need a common parent class**. If an object has the method we need, it works. This idea is called **duck typing**. Notice that `Landsat8`, `Sentinel2` and `Modis` above have **no common parent**, yet the code works.

In [4]:
class Kokil:
    def sing(self):
        return "Kuhu kuhu"

class BaulSinger:
    def sing(self):
        return "Ektara music..."

class Radio:
    def sing(self):
        return "FM 93.5 playing"

for thing in [Kokil(), BaulSinger(), Radio()]:
    print(thing.sing())

Kuhu kuhu
Ektara music...
FM 93.5 playing


## **Making Our Own Class Work with len() and Loops**
If our class has `__len__` and `__getitem__`, Python treats it like a list.

In [5]:
class Route:
    def __init__(self, *stops):
        self._stops = list(stops)
    def __len__(self):
        return len(self._stops)
    def __getitem__(self, i):
        return self._stops[i]

r = Route("Esplanade", "Park Street", "Maidan", "Rabindra Sadan")
print(len(r))
print(r[0], r[-1])
for stop in r:                # Looping works automatically
    print(" -", stop)
print("Maidan" in r)

4
Esplanade Rabindra Sadan
 - Esplanade
 - Park Street
 - Maidan
 - Rabindra Sadan
True


## **No Method Overloading Like Java**
In some languages, we can write the same method many times with different numbers of inputs. In Python, **the last one wins**. We use default values or `*args` instead.

In [6]:
class Calculator:
    def add(self, *nums):
        return sum(nums)

c = Calculator()
print(c.add(1, 2), c.add(1, 2, 3, 4))

3 10


## **Practice Problem: Areas of Different Shapes**

In [7]:
import math

class Circle:
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r ** 2

class Rectangle:
    def __init__(self, w, h): self.w, self.h = w, h
    def area(self): return self.w * self.h

class Triangle:
    def __init__(self, b, h): self.b, self.h = b, h
    def area(self): return 0.5 * self.b * self.h

shapes = [Circle(3), Rectangle(4, 5), Triangle(6, 2)]
for s in sorted(shapes, key=lambda s: s.area()):
    print(f"{type(s).__name__:<10}{s.area():7.2f}")

Triangle     6.00
Rectangle   20.00
Circle      28.27


In [8]:
# Quick question
# Is this polymorphism? What will it print?
# for x in ["ab", [1, 2, 3], (4,)]:
#     print(len(x))